In [3]:
import LowLevelFEM as FEM
using LowLevelFEM

gmsh.initialize();

In [4]:
gmsh.open("U-acel.geo")

mat1 = FEM.material("body", E=2e5, ν=0.3)
problem = FEM.Problem([mat1], type=:Solid)

Info    : Reading 'U-acel.geo'...
Info    : Meshing 1D...                                                                                                                         
Info    : [  0%] Meshing curve 9 (Line)
Info    : [ 10%] Meshing curve 10 (Line)
Info    : [ 20%] Meshing curve 11 (Line)
Info    : [ 30%] Meshing curve 12 (Line)
Info    : [ 40%] Meshing curve 13 (Line)
Info    : [ 50%] Meshing curve 14 (Line)
Info    : [ 60%] Meshing curve 15 (Line)
Info    : [ 60%] Meshing curve 16 (Line)
Info    : [ 70%] Meshing curve 17 (Line)
Info    : [ 80%] Meshing curve 18 (Line)
Info    : [ 90%] Meshing curve 19 (Line)
Info    : [100%] Meshing curve 20 (Line)
Info    : Done meshing 1D (Wall 0.000937018s, CPU 0.00086s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Transfinite)
Info    : [ 40%] Meshing surface 2 (Transfinite)
Info    : [ 70%] Meshing surface 3 (Transfinite)
Info    : Done meshing 2D (Wall 0.000453991s, CPU 0.000453s)
Info    : 965 nodes 1178 elements
Info

Problem("U-acel", :Solid, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 97465, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)

In [5]:
#gmsh.fltk.run()

In [6]:
xs = (200 * 8 * 4 + 92 * 8 * 54 * 2) / (2 * 92 * 8 + 200 * 8)
ys = 100

100

In [7]:
function Fx(x, y, z)
    ξ = x - xs
    η = y - ys
    φ = atan(η, ξ)
    r = √(ξ^2 + η^2)
    return -r * sin(φ) * 0.01
end

function Fy(x, y, z)
    ξ = x - xs
    η = y - ys
    φ = atan(η, ξ)
    r = √(ξ^2 + η^2)
    return r * cos(φ) * 0.01
end

Fy (generic function with 1 method)

In [8]:
supp = FEM.displacementConstraint("supp", ux=0, uy=0, uz=0)
load = FEM.displacementConstraint("load", ux=Fx, uy=Fy)
load0 = FEM.load("load", fy=-1);

In [9]:
K = FEM.stiffnessMatrix(problem)

sparse([1, 2, 3, 79, 80, 81, 220, 221, 222, 1225  …  292095, 292096, 292097, 292098, 292390, 292391, 292392, 292393, 292394, 292395], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  292395, 292395, 292395, 292395, 292395, 292395, 292395, 292395, 292395, 292395], [94017.09401709348, -32051.282051281574, -32051.282051281858, 21367.521367520545, 6410.256410256307, -16025.64102564079, -42735.042735042196, -6410.256410256547, -6410.256410256204, 21367.52136752102  …  -68376.06837606852, -1.4551915228366852e-11, -1.673470251262188e-10, 85470.08547008588, -1.4551915228366852e-11, -1.6643753042444587e-10, -170940.1709401722, -5.4569682106375694e-11, -3.1650415621697903e-10, 752136.752136753], 292395, 292395)

In [10]:
f = FEM.loadVector(problem, [load0]);

In [11]:
FEM.applyBoundaryConditions!(problem, K, f, [supp]);

LoadError: MethodError: no method matching applyBoundaryConditions!(::Problem, ::SystemMatrix, ::VectorField, ::Vector{BoundaryCondition})
The function `applyBoundaryConditions!` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  applyBoundaryConditions!([91m::Union{ScalarField, TensorField, VectorField}[39m, [91m::Vector{BoundaryCondition}[39m)
[0m[90m   @[39m [35mLowLevelFEM[39m [90m~/Dokumentumok/GitHub/LowLevelFEM.jl/src/[39m[90m[4mlinear.jl:3522[24m[39m


In [ ]:
q = FEM.solveDisplacement(K, f);

In [ ]:
S = FEM.solveStress(problem, q);

In [ ]:
u = FEM.showDoFResults(problem, q, "uvec", name="uvec", visible=false)
ux = FEM.showDoFResults(problem, q, "ux", name="ux", visible=false)
uy = FEM.showDoFResults(problem, q, "uy", name="uy", visible=false)
uz = FEM.showDoFResults(problem, q, "uz", name="uz", visible=false)

#s = FEM.showStressResults(problem, S, "s", name="σ", visible=true, smooth=false)
#ss = FEM.showStressResults(problem, S, "s", name="σ smooth", visible=false, smooth=true)
sx = FEM.showStressResults(problem, S, "sx", name="σx", visible=false, smooth=true)
sy = FEM.showStressResults(problem, S, "sy", name="σy", visible=false, smooth=true)
sz = FEM.showStressResults(problem, S, "sz", name="σz", visible=false, smooth=true)
sxy = FEM.showStressResults(problem, S, "sxy", name="τxy", visible=false, smooth=true)
syz = FEM.showStressResults(problem, S, "syz", name="τyz", visible=false, smooth=true)
szx = FEM.showStressResults(problem, S, "szx", name="τzx", visible=false, smooth=true)

Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)


10

In [ ]:
#FEM.plotOnPath(problem, "path7", syz, 1000, name="τyz", visible=true);

In [ ]:
gmsh.fltk.run()

-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib
FLTK version  : 1.3.8
OCC version   : 7.7.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------
Info    : Writing '/home/perebal/tmp/x_0'...
Info    : Writing VTK data in uvec_step0_level1_tol-0.0001_npart4/data0.vtu: fieldname = uvec

XOpenIM() failed
XRequest.18: BadValue 0x0


In [ ]:
gmsh.finalize()